In [1]:
import os
import asyncio
import resend
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool

load_dotenv(override=True)
resend.api_key = os.environ.get("RESEND_API_KEY")

In [2]:
# Ba phong cach viet email
instructions_chuyen_nghiep = """Bạn là sales agent của GrowthVN — công ty cung cấp
giải pháp marketing automation cho SME Việt Nam.
Viết email chuyên nghiệp, tập trung vào ROI, tiết kiệm thời gian, tăng doanh thu.
Tông: trang trọng, đáng tin cậy. Dưới 150 từ."""

instructions_than_thien = """Bạn là sales agent của GrowthVN — công ty cung cấp
giải pháp marketing automation cho SME Việt Nam.
Viết email gần gũi, đồng cảm với nỗi đau thực tế của chủ doanh nghiệp Việt Nam
(thiếu người, thiếu thời gian, cạnh tranh khốc liệt).
Tông: như bạn đồng nghiệp. Dưới 150 từ."""

instructions_ngan_gon = """Bạn là sales agent của GrowthVN — công ty cung cấp
giải pháp marketing automation cho SME Việt Nam.
Viết email tối đa 5 câu. Một câu hook mạnh. Ba câu giá trị. Một CTA rõ ràng.
Không câu nào thừa."""

agent_chuyen_nghiep = Agent(
    name="Sales Agent Chuyên Nghiệp",
    instructions=instructions_chuyen_nghiep,
    model="gpt-4o-mini"
)

agent_than_thien = Agent(
    name="Sales Agent Thân Thiện",
    instructions=instructions_than_thien,
    model="gpt-4o-mini"
)

agent_ngan_gon = Agent(
    name="Sales Agent Ngắn Gọn",
    instructions=instructions_ngan_gon,
    model="gpt-4o-mini"
)

# Biến mỗi Agent thành tool
mo_ta_tool = "Viết email sales giới thiệu dịch vụ GrowthVN"
tool1 = agent_chuyen_nghiep.as_tool(tool_name="email_chuyen_nghiep", tool_description=mo_ta_tool)
tool2 = agent_than_thien.as_tool(tool_name="email_than_thien", tool_description=mo_ta_tool)
tool3 = agent_ngan_gon.as_tool(tool_name="email_ngan_gon", tool_description=mo_ta_tool)

In [4]:
# Email Manager va cac tools

# tool viet tieu de
agent_tieu_de = Agent(
    name="Chuyên gia tiêu đề",
    instructions="""Viết tiêu đề email sales hấp dẫn cho doanh nghiệp Việt Nam. Tiêu đề dưới 60 ký tự, tạo sự tò mò hoặc nhấn vào lợi ích rõ ràng. Chỉ trả về tiêu đề, KHÔNG giải thích""",
    model="gpt-4o-mini"
)
tool_tieu_de = agent_tieu_de.as_tool(
    tool_name="viet_tieu_de",
    tool_description="Viết tiêu đề email sales hấp dẫn bằng tiếng Việt"
)

# tool chuyển HTML
agent_html = Agent(
    name="HTML Converter",
    instructions="""Chuyển nội dung email text/markdown sang HTML.
    Layout đơn giản, đọc tốt trên mobile. Dùng font-size 16px, line-height thoải mái.
    Thêm màu nhẹ nhàng cho tiêu đề nếu phù hợp. Chỉ trả về HTML, không giải thích.""",
    model="gpt-4o-mini"
)
tool_html = agent_html.as_tool(
    tool_name="chuyen_html",
    tool_description="Chuyển email text/markdown sang HTML format đẹp"
)

# tool gui email
@function_tool
def gui_html_email(tieu_de: str, noi_dung_html: str) -> dict:

    params = {
        "from": "onboarding@resend.dev",
        "to": ["thanhtam.udn@gmail.com"],
        "subject": tieu_de,
        "html": noi_dung_html,
    }

    email = resend.Emails.send(params)
    return {
        "status": "success",
        "email_id": email["id"]
    }

# Email Manager Agent - sẽ nhận Handoff
email_manager = Agent(
    name="Email Manager",
    instructions="""Bạn là Email Manager của GrowthVN.
Nhiệm vụ: nhận nội dung email đã chọn và xử lý hoàn chỉnh trước khi gửi.

Quy trình PHẢI tuân theo đúng thứ tự:
1. Gọi viet_tieu_de để tạo tiêu đề phù hợp với nội dung
2. Gọi chuyen_html để convert nội dung sang HTML đẹp
3. Gọi gui_html_email với tiêu đề và HTML vừa tạo để gửi đi

Không được bỏ qua bước nào. Không được tự ý viết lại nội dung.""",
    tools=[tool_tieu_de, tool_html, gui_html_email],
    model="gpt-4o-mini",
    handoff_description="Nhận email đã chọn, viết tiêu đề, chuyển HTML, và gửi qua Resend"

)

In [5]:
# Sales Manager - trung tâm điều phối

instructions_manager = """Bạn là Sales Manager của GrowthVN.

Quy trình làm việc:
1. Dùng CẢ BA tool (email_chuyen_nghiep, email_than_thien, email_ngan_gon) để tạo 3 phiên bản email
2. Đọc kỹ cả ba email, chọn phiên bản phù hợp nhất với đối tượng khách hàng được nhắc đến
3. Chuyển email đã chọn cho Email Manager để xử lý và gửi đi (handoff)

Tiêu chí chọn email:
- Phù hợp với ngữ cảnh và đối tượng nhận
- Rõ ràng, thuyết phục, không sáo rỗng
- Có CTA cụ thể

Bạn KHÔNG tự viết email. KHÔNG tự gửi email. Luôn dùng tools và handoff."""

sales_manager = Agent(
    name="Sales Manager GrowthVN",
    instructions=instructions_manager,
    tools=[tool1, tool2, tool3],
    handoffs=[email_manager],
    model="gpt-4o-mini"
)

In [6]:
async def chay_autoreach(yeu_cau: str):
    print(f"Yêu cầu: {yeu_cau}\n")

    with trace("autoreach-growthvn"):
        result = await Runner.run(sales_manager, yeu_cau)
    
    print(f"Kết quả: {result.final_output}")

await chay_autoreach("Gửi email sales giới thiệu GrowthVN đến giám đốc Marketing của các công ty SME Việt Nam")

Yêu cầu: Gửi email sales giới thiệu GrowthVN đến giám đốc Marketing của các công ty SME Việt Nam

Kết quả: Email đã được gửi thành công với tiêu đề **"Khám Phá Công Cụ Tăng Doanh Thu Hiệu Quả Nhất!"** đến giám đốc Marketing. 

Nếu bạn cần thêm thông tin hay cần hỗ trợ khác, hãy cho tôi biết nhé!
